In [9]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "BAAI/bge-small-zh-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}  # set True to compute cosine similarity
model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
    cache_folder="./cache"
)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39352.42it/s]
BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:

from milvus_vector import MilvusVectorSave
from markdown_parser import MarkdownParser

if __name__ == '__main__':
    # 解析文件内容
    file_path = r'./data/md/tech_report_0tfhhamx.md'
    parser = MarkdownParser(model)
    docs = parser.parse_markdown_to_documents(file_path)

    # 写入Milvus数据库
    mv = MilvusVectorSave(model)
    mv.create_collection()
    mv.create_connection()
    mv.add_documents(docs)

    client = mv.vector_store_saved.client
    # 得到表结构
    desc_collection = client.describe_collection(
        collection_name=COLLECTION_NAME
    )
    print('表结构是: ', desc_collection)

    # 得到当前表的，所有的index
    res = client.list_indexes(
        collection_name=COLLECTION_NAME
    )
    print('表中的所有索引：', res)

    if res:
        for i in res:
            # 得到索引的描述
            desc_index = client.describe_index(
                collection_name=COLLECTION_NAME,
                index_name=i
            )
            print(desc_index)

    result = client.query(
        collection_name=COLLECTION_NAME,
        filter="category == 'Title'",  # 查询 category == 'Title' 的所有数据
        output_fields=['text', 'category', 'filename']  # 指定返回的字段
    )

    print('测试 过滤查询的结果是: ', result)

20260411 09:41:54 | MainProcess | MainThread | markdown_parser.parse_markdown_to_documents:30 | INFO: 文件解析后的docs长度: 38
20260411 09:41:54 | MainProcess | MainThread | markdown_parser.parse_markdown_to_documents:34 | INFO: 文件合并后的长度: 16
20260411 09:41:54 | MainProcess | MainThread | markdown_parser.parse_markdown_to_documents:37 | INFO: 语义切割后的长度: 16
表结构是:  {'collection_name': 't_collection01', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 6000, 'enable_analyzer': 'true', 'analyzer_params': '{"tokenizer":"jieba","filter":["cnalphanumonly"]}'}}, {'field_id': 102, 'name': 'category', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 1000}}, {'field_id': 103, 'name': 'source', 'description': '', 'type': <DataTyp

In [14]:
client.query(
    collection_name=COLLECTION_NAME,
    filter="category == 'Title'",  # 查询 category == 'Title' 的所有数据
    output_fields=['text', 'category', 'filename']  # 指定返回的字段
)

data: ["{'text': '等离子体处理改善半导体材料表面特性的技术与方法', 'category': 'Title', 'filename': 'tech_report_0tfhhamx.md', 'id': 465532897672840450}", "{'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 半导体表面特性改善的具体应用方向', 'category': 'Title', 'filename': 'tech_report_0tfhhamx.md', 'id': 465532897672840452}", "{'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 工艺参数的关键影响因素', 'category': 'Title', 'filename': 'tech_report_0tfhhamx.md', 'id': 465532897672840456}", "{'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 先进等离子体表面处理技术', 'category': 'Title', 'filename': 'tech_report_0tfhhamx.md', 'id': 465532897672840459}", "{'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 表征技术与效果评估', 'category': 'Title', 'filename': 'tech_report_0tfhhamx.md', 'id': 465532897672840463}"], extra_info: {}

data: [], extra_info: {}